# Warm up
Let's start with some warm up exercises to get you familiar with the database and do some SQL querying.
Make sure to have downloaded the NBA dataset form Kaggle as described in the README.

In [ ]:
from nba.data_vis import (
    preprocess_data,
    plot_yearly_game_count,
    plot_yearly_home_win_rate,
    add_attendance_groups,
    add_crowd_situation,
    plot_home_win_rate_by_group,
)

In [ ]:
import sqlite3 as sql
import pandas as pd
from pathlib import Path

## TODO: Use pathlib to get the path to the data directory. Path(__file__) won't work in a Jupyter notebook.
DATA_PATH = Path.cwd() / 'data'
DATA_SQL_PATH = DATA_PATH / 'nba.sqlite'

## Create a SQLite connection and run some SQL queries

In [ ]:
con = sql.connect(DATA_SQL_PATH)  # connect to the database
query = 'SELECT * FROM game LIMIT 10'  # write a query that selects the first 10 rows from the game table
top_10_game = pd.read_sql(query, con)  # read the query into a pandas dataframe
top_10_game.head()  # print the dataframe

In [ ]:
# let's look at the schema of the database
# TODO: sqlite_master is the name of the table you want to inspeact: Replace TABLE_NAME with the name of the table you want to inspect
query = """
    SELECT *
    FROM sqlite_master
"""

pd.read_sql(query, con).sort_values('name')

In [ ]:
# let's see all the tables we have in the database
# TODO: Filter for tables only
query = """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
"""
pd.read_sql(query, con)

In [ ]:
# let's have a look at the columns in the game table
query = """
    PRAGMA table_info(draft_combine_stats)
"""
pd.read_sql(query, con)

In [ ]:
# let's find the earliest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date
    LIMIT 1
"""

print(pd.read_sql(query, con))

In [ ]:
# let's find the latest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date DESC
    LIMIT 1
"""

print(pd.read_sql(query, con))

In [ ]:
# TODO: Find four interesting facts about the data and write a query to find the answer

In [ ]:
# Insight 1: Yearly game count

## Load and preprocess data
query = """
    SELECT game_date, wl_home
    FROM game
"""
wl_game = pd.read_sql(query, con)  # read the query into a pandas dataframe
wl_game = preprocess_data(wl_game)  # preprocess the data

plot_yearly_game_count(wl_game)  # plot the yearly game count

- The annual number of games shows a general upward trend, consistently exceeding 500 since 1978.

In [ ]:
# Insight 2: Home team win rate

wl_game_recent = wl_game[wl_game['year'] >= 1978] # filter for games from 1978 onwards
plot_yearly_home_win_rate(wl_game_recent)  # plot the yearly home win rate

- The home team win rate remains consistently above 50%, but has shown an overall downward trend since 1978.

In [ ]:
# Insight 3: Home advantage and crowd size (2010-2019)

## Load data: home result + attendance for regular-season games, seasons 2010-11 to 2018-19
query = """
    SELECT DISTINCT
        g.game_id,
        g.season_id,
        g.wl_home,
        i.attendance
    FROM game AS g
    JOIN game_info AS i
        ON g.game_id = i.game_id
    WHERE g.game_id LIKE '002%'
        AND g.season_id BETWEEN '22010' AND '22018'
        AND i.attendance > 0
        AND i.attendance <= 30000
        AND g.wl_home IN ('W', 'L')
"""
home_crowd = pd.read_sql(query, con)  # read the query into a pandas dataframe
home_crowd = add_attendance_groups(home_crowd)  # add attendance groups

plot_home_win_rate_by_group(
    home_crowd,
    "attendance_group",
    "Home advantage grows with crowd size (2010-2019)",
    "Attendance (spectators per game)",
)

- Home advantage (win rate above 50%) seems to increase with crowd size: in regular-season games from 2010-11 to 2018-19, home teams won about 52-53% of games with fewer than 17,500 spectators, but about 62% of games with more than 17,500 spectators.

In [ ]:
# Insight 4: COVID natural experiment - home advantage without fans

## Load data: all regular-season games of 2015-16 to 2018-19 and 2020-21
## LEFT JOIN keeps games without attendance data (= empty arenas in 2020-21)
query = """
    SELECT DISTINCT
        g.game_id,
        g.season_id,
        g.wl_home,
        i.attendance
    FROM game AS g
    LEFT JOIN game_info AS i
        ON g.game_id = i.game_id
    WHERE g.game_id LIKE '002%'
        AND g.season_id IN ('22015', '22016', '22017', '22018', '22020')
        AND g.wl_home IN ('W', 'L')
"""
home_covid = pd.read_sql(query, con)  # read the query into a pandas dataframe
home_covid = add_crowd_situation(home_covid)  # label: normal / fans / no fans

plot_home_win_rate_by_group(
    home_covid,
    "situation",
    "Without fans, home advantage disappears",
    "",
)

- The Covid-19 season 2020-21 works as a natural experiment: many games were played in empty arenas for reasons unrelated to team quality, while some games still had (limited) crowds.
- Without fans, the home advantage practically disappeared: home teams won only 50.4% of games (a coin flip), compared with 60.4% in 2020-21 games with fans and 58.6% in the normal seasons 2015-16 to 2018-19.